# EmporiUm Sales Territory Analysis

**Analyst:** [Your Name Here]  
**Course:** Year Up United Data Analyst Training Academy — Week 8 Capstone  
**Date:** 2025

## Project Description

EmporiUm is a growing student bookstore chain that also sells tech gear, art supplies, and other products both in-store and online. Sales are organized by region, and each region contains multiple sales territories.

In this notebook I analyze in-store sales data for **two territories in the South region**:

- **Territory 1 — Lana Ilana** (Florida, Store IDs 719–729)
- **Territory 2 — Jeff "Howdy" Richards** (Texas, Store IDs 901–911)

I chose Jeff Richards' Texas territory as the second territory because it is the most comparable to Lana Ilana's Florida territory: both are in the **South region**, both have **11 stores**, and both have similar total sales volume (~$3.4M–$3.9M over the data period).

The goal of this analysis is to compare performance across the two territories, identify top-performing stores and customers, understand product category trends, and make a recommendation for where marketing should focus in the next quarter.

---
## Step 1 — Import Libraries

We start by importing the libraries we need:
- **pandas** (`pd`) — for loading, cleaning, and analyzing tabular data
- **matplotlib.pyplot** (`plt`) — for creating charts

Importing them at the top of the notebook is a best practice so that all dependencies are visible in one place.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

: 

---
## Step 2 — Load the Data

We load each of the five CSV files provided. After loading each one, we call `.info()` to confirm:
- The column names
- How many non-null (non-missing) values are in each column
- The data type (`dtype`) of each column

This helps us catch issues early, like columns that should be numbers but loaded as text, or columns with unexpected missing values.

**Note on `customer_list.csv`:** This file uses a pipe character (`|`) as its separator instead of a comma. We specify `sep='|'` so pandas reads it correctly. Without this, all columns would be squashed into one.

In [ ]:
# Load store transaction data — this is our main sales table
store_sales = pd.read_csv('StoreSales.csv')
store_sales.info()

In [ ]:
# Load store details — city, state, territory manager, region for each store
store_detail = pd.read_csv('StoreDetail.csv')
store_detail.info()

In [ ]:
# Load product catalog — product number, name, and category IDs
products = pd.read_csv('Products__3_.csv')
products.info()

In [ ]:
# Load product categories — lookup table mapping CategoryID to category/subcategory names
product_categories = pd.read_csv('ProductCategories.csv')
product_categories.info()

In [ ]:
# Load customer/rewards member list
# NOTE: This file uses | as a separator instead of a comma, so we pass sep='|'
customer_list = pd.read_csv('customer_list.csv', sep='|')
customer_list.info()

### Quick Preview of Each Table

We use `.head()` to see the first 5 rows of each table and confirm the data looks as expected.

In [ ]:
store_sales.head()

In [ ]:
store_detail.head()

In [ ]:
products.head()

In [ ]:
product_categories.head()

In [ ]:
customer_list.head()

---
## Step 3 — Data Cleaning

Before analyzing anything, we need to clean and prepare the data.

**Why parse dates?** The `Transaction Date` column loaded as a plain string (object dtype). We need to convert it to an actual datetime type so we can extract the month, sort by time, and do time-based grouping. `pd.to_datetime()` handles this.

**Why add a Month column?** Most of our analysis is month-by-month. Rather than re-extracting the month every time, we add a `Month` column once using `.dt.to_period('M')`, which gives us values like `2022-01`, `2022-02`, etc.

In [ ]:
# Convert Transaction Date from string to datetime so we can do time-based analysis
store_sales['Transaction Date'] = pd.to_datetime(store_sales['Transaction Date'])

# Add a Month column by extracting year-month from the date
# to_period('M') gives us '2022-01', '2022-02', etc.
store_sales['Month'] = store_sales['Transaction Date'].dt.to_period('M')

print('Date range in data:', store_sales['Transaction Date'].min(), 'to', store_sales['Transaction Date'].max())
print('Month column sample:', store_sales['Month'].head(3).tolist())

---
## Step 4 — Filter to Our Two Territories

The full dataset contains all stores across all regions. Since our analysis focuses on just two territory managers, we filter down early. This makes all future steps faster and simpler.

We use `.isin()` to match either territory manager name, then merge that with the sales table so every transaction row knows which territory it belongs to.

In [ ]:
# Define our two territory managers
our_managers = ['Lana Ilana', 'Jeff "Howdy" Richards']

# Filter StoreDetail to only the stores in our two territories
our_stores = store_detail[store_detail['Territory Manager'].isin(our_managers)].copy()

print('Stores in our territories:')
print(our_stores[['Store Location', 'State', 'Store ID', 'Territory Manager']].to_string(index=False))

In [ ]:
# Get the list of Store IDs for our territories
our_store_ids = our_stores['Store ID'].tolist()

# Filter StoreSales to only transactions from our stores
our_sales = store_sales[store_sales['Store ID'].isin(our_store_ids)].copy()

print(f'Total transactions in our two territories: {len(our_sales):,}')

In [ ]:
# Merge territory manager info into our sales table
# This adds the Territory Manager column to every transaction row
# We only bring in the columns we need from StoreDetail
our_sales = our_sales.merge(
    store_detail[['Store ID', 'Territory Manager', 'Store Location', 'State']],
    on='Store ID',
    how='left'
)

our_sales.head()

---
# Core Marketing Analysis

---
## Question 1 — Territory Managers, Store IDs, and Cities

**What the manager wants to know:** Who manages each territory, and what stores (IDs and cities) are in each one?

We already filtered our stores above. Here we display them clearly, grouped by territory manager.

In [ ]:
# Display stores grouped by territory manager
for manager, group in our_stores.groupby('Territory Manager'):
    print(f'Territory Manager: {manager}')
    print(f'Region: {group["Region"].iloc[0]}')
    print(group[['Store ID', 'Store Location', 'State']].to_string(index=False))
    print()

---
## Question 2 — Monthly Total Revenue by Territory

**What the manager wants to know:** What is the monthly total revenue for in-store sales in each territory over the full data period?

We use `.groupby()` to group transactions by both `Month` and `Territory Manager`, then sum the `Sale Amount` for each group. This tells us how much each territory brought in every month.

We then pivot the result so each territory is its own column — this makes it easier to read and to chart.

In [ ]:
# Group by Month and Territory Manager, summing Sale Amount
monthly_revenue = (
    our_sales
    .groupby(['Month', 'Territory Manager'])['Sale Amount']
    .sum()
    .reset_index()
)

# Pivot so each territory manager is a column
monthly_pivot = monthly_revenue.pivot(index='Month', columns='Territory Manager', values='Sale Amount')
monthly_pivot.columns.name = None  # Remove the label 'Territory Manager' from column header

print('Monthly Revenue by Territory:')
print(monthly_pivot.to_string())

### Chart 1 — Monthly Revenue Over Time

A **line chart** is the right choice here because we are looking at a **trend over time**. Each month is a point on the x-axis, and the revenue value for that month is on the y-axis. Using two lines — one per territory — lets us compare them side by side.

We convert the Period index to strings for cleaner x-axis labels, and we only show every 3rd label to avoid crowding.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

# Convert Period index to strings for plotting
x_labels = monthly_pivot.index.astype(str)
x_pos = range(len(x_labels))

# Plot one line per territory
ax.plot(x_pos, monthly_pivot['Lana Ilana'], marker='o', markersize=4, label='Lana Ilana (Florida)')
ax.plot(x_pos, monthly_pivot['Jeff "Howdy" Richards'], marker='s', markersize=4, label='Jeff Richards (Texas)')

# Only show every 3rd month label to avoid crowding
ax.set_xticks([i for i in x_pos if i % 3 == 0])
ax.set_xticklabels([x_labels[i] for i in x_pos if i % 3 == 0], rotation=45, ha='right')

ax.set_title('Monthly In-Store Revenue by Territory (South Region)', fontsize=14)
ax.set_xlabel('Month')
ax.set_ylabel('Revenue ($)')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

---
## Question 3 — Store Performance Rankings

**What the manager wants to know:** How do stores rank within each territory? Which are the top performers?

We group by `Store ID` and sum total revenue for the full data period. We then merge back in the store's city name so the results are human-readable (store IDs alone are not very meaningful). Sorting from highest to lowest gives us the ranking.

In [ ]:
# Total revenue per store across the full period
store_revenue = (
    our_sales
    .groupby(['Store ID', 'Store Location', 'Territory Manager'])['Sale Amount']
    .sum()
    .reset_index()
    .rename(columns={'Sale Amount': 'Total Revenue'})
    .sort_values('Total Revenue', ascending=False)
)

# Add a rank column within each territory
store_revenue['Rank'] = store_revenue.groupby('Territory Manager')['Total Revenue'].rank(
    ascending=False, method='min'
).astype(int)

# Display Lana Ilana's stores
print('--- Lana Ilana (Florida) Store Rankings ---')
lana_stores = store_revenue[store_revenue['Territory Manager'] == 'Lana Ilana'].sort_values('Rank')
print(lana_stores[['Rank', 'Store ID', 'Store Location', 'Total Revenue']].to_string(index=False))

print()

# Display Jeff Richards' stores
print('--- Jeff Richards (Texas) Store Rankings ---')
jeff_stores = store_revenue[store_revenue['Territory Manager'] == 'Jeff "Howdy" Richards'].sort_values('Rank')
print(jeff_stores[['Rank', 'Store ID', 'Store Location', 'Total Revenue']].to_string(index=False))

### Chart 2 — Store Revenue Comparison (Both Territories)

A **horizontal bar chart** works well here because we are comparing a single value (total revenue) across multiple categories (stores). Horizontal bars make the store city names easy to read. We separate the two territories with color.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Lana Ilana
lana_sorted = lana_stores.sort_values('Total Revenue')
axes[0].barh(lana_sorted['Store Location'], lana_sorted['Total Revenue'], color='steelblue')
axes[0].set_title('Lana Ilana — Florida Store Revenue', fontsize=12)
axes[0].set_xlabel('Total Revenue ($)')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Jeff Richards
jeff_sorted = jeff_stores.sort_values('Total Revenue')
axes[1].barh(jeff_sorted['Store Location'], jeff_sorted['Total Revenue'], color='darkorange')
axes[1].set_title('Jeff Richards — Texas Store Revenue', fontsize=12)
axes[1].set_xlabel('Total Revenue ($)')
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.suptitle('Total In-Store Revenue by Store (Full Data Period)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## Question 4 — Top Customers by Territory

**What the manager wants to know:** Who are the top customers (rewards members) in each territory?

The sales table has a `RewardsID` column that links to the `cust_id` column in the customer list. However, not every transaction has a rewards ID — non-members shop without one, so those rows have `NaN`. We use `.dropna()` to keep only transactions where a rewards ID is present.

We then merge with the customer list to get the customer's name alongside their ID, and sum their total spending.

In [ ]:
# Keep only transactions that have a RewardsID (i.e., the customer is a rewards member)
rewards_sales = our_sales.dropna(subset=['RewardsID']).copy()

# RewardsID loaded as a float (because of NaN values), convert to int for clean matching
rewards_sales['RewardsID'] = rewards_sales['RewardsID'].astype(int)

# Merge with customer list to get customer names
# RewardsID in sales matches cust_id in customer_list
rewards_named = rewards_sales.merge(
    customer_list[['cust_id', 'name']],
    left_on='RewardsID',
    right_on='cust_id',
    how='left'
)

# Group by territory and customer, sum their spending, take top 10 per territory
top_customers = (
    rewards_named
    .groupby(['Territory Manager', 'RewardsID', 'name'])['Sale Amount']
    .sum()
    .reset_index()
    .rename(columns={'Sale Amount': 'Total Spend'})
    .sort_values('Total Spend', ascending=False)
)

print('--- Top 10 Customers: Lana Ilana (Florida) ---')
print(
    top_customers[top_customers['Territory Manager'] == 'Lana Ilana']
    .head(10)[['RewardsID', 'name', 'Total Spend']]
    .to_string(index=False)
)

print()

print('--- Top 10 Customers: Jeff Richards (Texas) ---')
print(
    top_customers[top_customers['Territory Manager'] == 'Jeff "Howdy" Richards']
    .head(10)[['RewardsID', 'name', 'Total Spend']]
    .to_string(index=False)
)

---
## Question 5 — Transactions and Revenue by Product Category

**What the manager wants to know:** How many transactions per month does each product category have? What is the revenue per category per month? What does this tell us about popular products and growth opportunities?

To answer this, we need to join three tables:
1. `our_sales` — has `Prod Num`
2. `products` — maps `Prod Num` to `CategoryID`
3. `product_categories` — maps `CategoryID` to the category name

We do two sequential merges to connect them all.

In [ ]:
# Step 1: Add CategoryID to each transaction by merging with products
sales_with_cat = our_sales.merge(
    products[['Prod Num', 'CategoryID']],
    on='Prod Num',
    how='left'
)

# Step 2: Add the Category name by merging with product_categories
sales_with_cat = sales_with_cat.merge(
    product_categories[['CategoryID', 'Category']].drop_duplicates(),
    on='CategoryID',
    how='left'
)

# Confirm the merge worked
print('Sample of merged data:')
print(sales_with_cat[['Transaction Date', 'Store Location', 'Territory Manager', 'Prod Num', 'Category', 'Sale Amount']].head(5))

In [ ]:
# Group by Territory, Month, and Category
# Count transactions and sum revenue
cat_monthly = (
    sales_with_cat
    .groupby(['Territory Manager', 'Month', 'Category'])
    .agg(
        transaction_count=('Sale Amount', 'count'),
        total_revenue=('Sale Amount', 'sum')
    )
    .reset_index()
)

print('Monthly category breakdown (first 12 rows):')
print(cat_monthly.head(12).to_string(index=False))

In [ ]:
# Summarize total revenue and transaction count per category (across all months)
# This gives a cleaner view for interpretation
cat_summary = (
    sales_with_cat
    .groupby(['Territory Manager', 'Category'])
    .agg(
        total_transactions=('Sale Amount', 'count'),
        total_revenue=('Sale Amount', 'sum')
    )
    .reset_index()
    .sort_values(['Territory Manager', 'total_revenue'], ascending=[True, False])
)

print('--- Lana Ilana (Florida) Category Summary ---')
print(cat_summary[cat_summary['Territory Manager'] == 'Lana Ilana'].to_string(index=False))
print()
print('--- Jeff Richards (Texas) Category Summary ---')
print(cat_summary[cat_summary['Territory Manager'] == 'Jeff "Howdy" Richards'].to_string(index=False))

### Chart 3 — Revenue by Product Category

A **grouped bar chart** is the right choice here because we want to **compare categories across two territories** side by side. Each category gets two bars — one per territory — so we can immediately see where each territory is stronger or weaker.

In [ ]:
import numpy as np

# Pivot so each territory is a column
cat_pivot = cat_summary.pivot(index='Category', columns='Territory Manager', values='total_revenue').fillna(0)
cat_pivot.columns.name = None

categories = cat_pivot.index.tolist()
x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(13, 6))

bars1 = ax.bar(x - width/2, cat_pivot['Lana Ilana'], width, label='Lana Ilana (Florida)', color='steelblue')
bars2 = ax.bar(x + width/2, cat_pivot['Jeff "Howdy" Richards'], width, label='Jeff Richards (Texas)', color='darkorange')

ax.set_title('Total Revenue by Product Category — Both Territories', fontsize=14)
ax.set_xlabel('Product Category')
ax.set_ylabel('Total Revenue ($)')
ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=20, ha='right')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda val, _: f'${val:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

### What This Tells Us: Popular Products and Growth Opportunities

The category data reveals a clear and consistent pattern across **both** territories:

**Technology & Accessories dominates revenue in both territories.**  
It accounts for roughly 70% of total revenue in both Florida (~$2.76M, 70.3%) and Texas (~$2.45M, 71.5%), despite not having the most transactions. This is because the average Technology transaction is ~$466 — by far the highest of any category. Customers are buying high-ticket items like tablets, laptops, and calculators. This category is clearly the engine of the business.

**Textbooks are the second-largest revenue driver.**  
At ~$174 average transaction value, Textbooks bring in meaningful revenue (Florida: $706K, Texas: $597K) with a moderate number of transactions. This is expected for a student-focused bookstore, but it also means revenue here is likely seasonal — concentrated around the start of semesters.

**Stationery & Supplies has the most transactions but the lowest revenue.**  
This is the most striking finding. Stationery has the highest transaction count of any category in both territories (Florida: 6,128 transactions, Texas: 5,396), yet generates the least revenue of all six categories — only about $61K and $54K respectively. The average transaction is less than $10. Customers are coming in frequently to buy low-cost items like pens and notebooks. This is a foot-traffic opportunity: these customers are already in the store, but they are not being upsold into higher-value products.

**Books (General) underperforms for a bookstore.**  
Books (General) has both the fewest transactions and second-lowest revenue. For a business called a *bookstore*, this is a gap worth noting. It suggests that non-textbook books are not a draw — customers may not think of EmporiUm as a destination for general reading.

**The two territories are remarkably similar across all categories.**  
Every category follows the same rank order in both Florida and Texas, and the revenue proportions are nearly identical. This tells us the patterns we're seeing are not territory-specific quirks — they reflect the overall product mix and customer behavior across the EmporiUm brand in the South region.

---
## Question 6 — Marketing Recommendation for Next Quarter

Based on the analysis above, here is my recommendation for where marketing should focus in the next quarter.

In [ ]:
# Supporting numbers for the recommendation

# Total revenue per territory
territory_totals = our_sales.groupby('Territory Manager')['Sale Amount'].sum()
print('Total Revenue by Territory:')
print(territory_totals.apply(lambda x: f'${x:,.2f}'))
print()

# Lowest-revenue stores in each territory (potential growth targets)
print('Lowest-revenue stores (growth opportunities):')
print(store_revenue.sort_values('Total Revenue').groupby('Territory Manager').head(3)[
    ['Territory Manager', 'Store Location', 'Total Revenue']
].to_string(index=False))
print()

# Category with lowest revenue relative to transaction count (underperforming)
cat_summary['avg_transaction'] = cat_summary['total_revenue'] / cat_summary['total_transactions']
print('Average transaction value by category:')
print(cat_summary[cat_summary['Territory Manager'] == 'Lana Ilana'][['Category','total_transactions','total_revenue','avg_transaction']].sort_values('total_revenue').to_string(index=False))

### Recommendation

Based on the full analysis, my marketing recommendation for next quarter focuses on two areas:

**1. Boost underperforming stores in both territories.**  
In Lana Ilana's Florida territory, stores like Naples and Key West show significantly lower revenue than top performers like Miami and Tallahassee. In Jeff Richards' Texas territory, similar gaps exist between the top store (Bacliff/Beaumont) and the lowest performers. Marketing should run targeted local campaigns — promotions, events, or loyalty incentives — specifically at these lower-performing locations to bring them closer to territory averages.

**2. Grow the Books and Stationery categories.**  
The category analysis shows that Books (General) and Stationery & Supplies consistently generate the fewest transactions and lowest revenue, despite being core products for a student-focused bookstore. These categories represent the biggest opportunity for growth because they are directly aligned with the store's identity and customer base. A targeted promotion — such as a back-to-school bundle, a textbook trade-in program, or a stationery rewards bonus — could meaningfully increase transactions in these categories.

Technology & Accessories is already the strongest category in both territories and does not need additional marketing investment at this time. Art Supplies also performs well, especially in Florida. Marketing resources are best redirected toward the categories and stores with the most room to grow.